# Mapas de referencia


In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
# import netCDF4 as nc

# # Open, read data, and close the file
# ds = nc.Dataset("DATA/buoys.nc", "r")
# G = ds.variables["longitude"][:].compressed() -360
# L = ds.variables["latitude"][:].compressed()
# id_coq = np.where(L>-31)
# id_val = np.where(L>=-31)
# coords_val = np.array((G[id_val], L[id_val]))
# coords_coq = np.array((G[id_coq], L[id_coq]))
# ds.close()

# # Guardar
# np.save("DATA/val_buoys.npy", coords_val)
# np.save("DATA/coq_buoys.npy", coords_coq)

In [3]:
coords = np.load('DATA/coq_buoys.npy')
lat = coords[1]
lon = coords[0]

print(f'Latitudes: {lat[:]}')
print(f'Longitudes: {lon[:]}')

Latitudes: [-29.960938 -29.926758 -29.918945 -29.910156 -29.893555 -29.885742
 -29.860352 -29.83496  -29.94336  -29.893555 -29.86914  -29.85254
 -29.83496  -29.785156]
Longitudes: [-71.3291   -71.27832  -71.34766  -71.33984  -71.33984  -71.331055
 -71.31445  -71.30664  -71.42285  -71.40625  -71.38965  -71.37305
 -71.34766  -71.37305 ]


## 2. Funciones para Graficar

In [6]:
#Configuraciones

# URL del servicio WMS de GEBCO
GEBCO_WMS_URL = "https://www.gebco.net/data_and_products/gebco_web_services/web_map_service/mapserv"
GEBCO_LAYER   = "GEBCO_LATEST"          # capa de batimetría + topografía

# Extensión del mapa [lon_min, lon_max, lat_min, lat_max]
f = .25
EXTENT = [min(coords[0])-f, max(coords[0])+f, min(coords[1])-f, max(coords[1])+f]

# tamaño figura
FIG_SIZE = (8,8)

# Ciudades de referencia
coordenadas_coquimbo = np.array([-29.90777, -71.25416])
valparaiso_coord = np.array([-33.0461, -71.6197])
ciudades = {
            'Valparaiso': valparaiso_coord,
            'Coquimbo'  : coordenadas_coquimbo
           }

In [7]:
EXTENT

[np.float32(-71.67285),
 np.float32(-71.02832),
 np.float32(-30.210938),
 np.float32(-29.535156)]

In [ ]:
#Configuraciones

# URL del servicio WMS de GEBCO
GEBCO_WMS_URL = "https://www.gebco.net/data_and_products/gebco_web_services/web_map_service/mapserv"
GEBCO_LAYER   = "GEBCO_LATEST"          # capa de batimetría + topografía

# Extensión del mapa [lon_min, lon_max, lat_min, lat_max]
f = .25
EXTENT = [min(coords[0])-f, max(coords[0])+f, min(coords[1])-f, max(coords[1])+f]

# tamaño figura
FIG_SIZE = (8,8)

# Ciudades de referencia
coordenadas_coquimbo = np.array([-29.90777, -71.25416])
#valparaiso_coord = np.array([-33.0461, -71.6197])
ciudades = {
            'Valparaiso': valparaiso_coord,
            'Coquimbo'  : coordenadas_coquimbo
           }
def graficar(coords, ref='Coquimbo'):

    proj = ccrs.PlateCarree()

    fig, ax = plt.subplots(figsize=FIG_SIZE, subplot_kw={"projection": proj})
    ax.set_extent(EXTENT, crs=proj)

    # — Fondo WMS GEBCO —
    try:
        # wms_url = 'https://wms.gebco.net/2025/mapserv?'
        # wms_layer = 'GEBCO_2025'
        # ax.add_wms(wms=wms_url, layers=[wms_layer])
        # print("✓ WMS GEBCO cargado correctamente.")

        wms_url = "https://landsat2.arcgis.com:443/arcgis/services/Landsat/MS/ImageServer/WMSServer?request=GetLegendGraphic%26version=1.3.0%26format=image/png%26layer=MS:Bathymetric with DRA"
        wms_layer = 'MS:Bathymetric with DRA' 
        ax.add_wms(wms=wms_url, layers=[wms_layer])
        print("✓ Natural Earth cargado correctamente.")

    except Exception as e:
        print(f"⚠ No se pudo cargar el WMS de GEBCO: {e}")
        print("  Usando fondo alternativo (Natural Earth).")
        ax.stock_img()

    # — Líneas de costa y bordes de países (encima del WMS) —
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="white", zorder=3)
    #ax.add_feature(cfeature.BORDERS,   linewidth=0.2, edgecolor="white", alpha=0.5, zorder=3)

    # — Grilla —
    gl = ax.gridlines(
        draw_labels=True, linewidth=0.4,
        color="white", alpha=0.5, linestyle="--"
    )
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8, "color": "white"}
    gl.ylabel_style = {"size": 8, "color": "white"}

    # — Boyas —
    ax.plot(coords[0],coords[1], linestyle="none", transform=proj, marker="^", 
            color="red", markersize=7,  zorder=5, label='Boya')
    #  — Ciudad de referencia —
    if ref is not None:
        ax.plot(ciudades[ref][1],ciudades[ref][0], linestyle="none", transform=proj, marker="*", 
                color="magenta", markersize=7,  zorder=5, label=ref)

    
    # — Leyenda —
    custom_buoy = mlines.Line2D([], [], color='red', marker='^',  linestyle=' ' ,
                          markersize=7, label='Boya')
    
    custom_ref = mlines.Line2D([], [], color='magenta', marker='*', linestyle=' ' ,
                          markersize=7, label=ref)
    handles = [custom_buoy, custom_ref ]
    ax.legend(
        handles=handles,
        loc="lower left",
        fontsize=9,
        framealpha=0.7,
        facecolor="#1a1a2e",
        labelcolor="white"
    )

    # — Título —
    # ax.set_title(
    #     "Red de Boyas DART — Cuenca del Pacífico",
    #     fontsize=13, color="white", pad=10
    # )

    #fig.patch.set_facecolor("#1a1a2e")
    #ax.set_facecolor("#1a1a2e")

    plt.tight_layout()
    figname="FIGS/boyas.png"
    plt.savefig(figname, dpi=300, bbox_inches="tight",facecolor=fig.get_facecolor())
    print(f"✓ Mapa guardado como {figname}")
    plt.show()

graficar(coords)

ImportError: OWSLib is required to use OGC web services.

## 3. Graficar

In [9]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches, matplotlib.lines as mlines
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, LightSource
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs, cartopy.feature as cfeature
from cartopy.io.img_tiles import GoogleTiles
from scipy.ndimage import gaussian_filter
import urllib.request, warnings
warnings.filterwarnings("ignore")

# ── DATOS ────────────────────────────────────────────────────────────────────
lat_pronostico = np.array([-29.960938])
lon_pronostico = np.array([-71.3291])

lats_boyas_raw = np.array([-29.918945, -29.910156, -29.893555,
                            -29.885742, -29.860352, -29.83496])
lons_boyas_raw = np.array([-71.34766,  -71.33984,  -71.33984,
                            -71.331055, -71.31445,  -71.30664])
orden = np.argsort(lats_boyas_raw)
lats_boyas = lats_boyas_raw[orden]
lons_boyas = lons_boyas_raw[orden]
numeros_boyas = np.arange(1, 7)

# ── CONFIGURACIÓN ────────────────────────────────────────────────────────────
PAD = 0.18
all_lats_orig = np.array([-29.960938,-29.926758,-29.918945,-29.910156,
                           -29.893555,-29.885742,-29.860352,-29.83496,
                           -29.94336, -29.893555,-29.86914, -29.85254,
                           -29.83496, -29.785156])
all_lons_orig = np.array([-71.3291,  -71.27832, -71.34766, -71.33984,
                           -71.33984, -71.331055,-71.31445, -71.30664,
                           -71.42285, -71.40625, -71.38965, -71.37305,
                           -71.34766, -71.37305])
ZOOM_EXT  = [all_lons_orig.min()-PAD, all_lons_orig.max()+PAD,
             all_lats_orig.min()-PAD, all_lats_orig.max()+PAD]
CHILE_EXT = [-76, -65, -56, -17]
COQUIMBO  = {"nombre": "Coquimbo", "lat": -29.90777, "lon": -71.25416}

BG = "#0d1b2a"; PANEL = "#0a1628"; FG = "#e8eef4"
ACCENT = "#00c8ff"; RECT_C = "#ffd166"
BOYA_C = "#ef476f"; REF_C  = "#06d6a0"; PRON_C = "#f9a825"

# Isobatas a dibujar y sus estilos
ISOBATAS   = [-200, -500, -1000, -2000]
ISO_COLOR  = "white"
ISO_ALPHA  = 0.35          # tenues
ISO_LW     = [0.8, 0.7, 0.9, 0.7]   # -200 un poco más gruesa (borde plataforma)
ISO_LS     = ["-", "--", "-", "--"]

TILE_URLS = {
    "ocean" : "https://server.arcgisonline.com/ArcGIS/rest/services/Ocean/World_Ocean_Base/MapServer/tile/{z}/{y}/{x}",
    "shaded": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Shaded_Relief/MapServer/tile/{z}/{y}/{x}",
}

COAST_PTS = np.array([
    [-30.976,-71.655],[-30.896,-71.680],[-30.697,-71.704],
    [-30.509,-71.703],[-30.285,-71.542],[-30.241,-71.638],
    [-30.163,-71.388],[-30.046,-71.382],[-29.979,-71.393],
    [-29.962,-71.380],[-29.941,-71.360],[-29.931,-71.351],
    [-29.836,-71.285],[-29.728,-71.336],[-29.666,-71.319],
    [-29.587,-71.312],[-29.472,-71.319],[-29.401,-71.315],
])

stroke = lambda lw: pe.withStroke(linewidth=lw, foreground="black")

def coast_lon(lat):
    return float(np.interp(lat, COAST_PTS[:,0], COAST_PTS[:,1]))

# ── TILES ────────────────────────────────────────────────────────────────────
class XYZTiles(GoogleTiles):
    def __init__(self, url_tpl, **kw): self._tpl=url_tpl; super().__init__(**kw)
    def _image_url(self, tile):
        x,y,z=tile; return self._tpl.format(z=z,x=x,y=y)

def tiles_ok(key):
    url = TILE_URLS[key].format(z=4,x=309,y=660)
    try:
        r = urllib.request.urlopen(
            urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"}), timeout=6)
        return r.status==200
    except: return False

def add_tiles(ax, key, zoom, alpha=1.0):
    ax.add_image(XYZTiles(TILE_URLS[key]), zoom, alpha=alpha,
                 interpolation="bilinear")

# ── BATIMETRÍA: descarga ETOPO1 o modelo sintético ───────────────────────────
def get_bathy_grid(ext, res_deg=0.01):
    """
    Intenta descargar batimetría real de ERDDAP (ETOPO1, ~1 arc-min).
    Si no hay acceso, genera el modelo sintético calibrado para esta zona.
    Devuelve (lon2d, lat2d, z2d) donde z2d < 0 es profundidad en metros.
    """
    W, E, S, N = ext
    # ── Intento 1: ERDDAP NOAA ETOPO1 ────────────────────────────
    try:
        # stride=1 → resolución nativa ~1/60°; pedimos un margen extra
        url = (f"https://coastwatch.pfeg.noaa.gov/erddap/griddap/etopo180.csv?"
               f"altitude[({S-0.05:.3f}):1:({N+0.05:.3f})][({W-0.05:.3f}):1:({E+0.05:.3f})]")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        raw = urllib.request.urlopen(req, timeout=15).read().decode()
        lines = raw.strip().split("\n")
        # ERDDAP CSV: línea 0 = nombres, línea 1 = unidades, resto = datos
        rows = []
        for l in lines[2:]:
            if not l.strip(): continue
            rows.append(list(map(float, l.split(","))))
        data = np.array(rows)   # columns: latitude, longitude, altitude(m)
        # ERDDAP devuelve: col 0=latitude, col 1=longitude, col 2=altitude
        lats_u = np.unique(data[:,0])
        lons_u = np.unique(data[:,1])
        Z = data[:,2].reshape(len(lats_u), len(lons_u))
        LON, LAT = np.meshgrid(lons_u, lats_u)
        print("  → Batimetría ETOPO1 (ERDDAP) descargada.")
        return LON, LAT, Z
    except Exception as e:
        print(f"  → ERDDAP no disponible ({e}).")
        # Diagnóstico: mostrar primeras líneas si el error es de parsing
        if "float" in str(e).lower() or "convert" in str(e).lower():
            try:
                raw2 = urllib.request.urlopen(
                    urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"}),
                    timeout=15).read().decode()
                print("  → Primeras 3 líneas de ERDDAP:")
                for l in raw2.strip().split("\n")[:3]:
                    print(f"     {l}")
            except: pass
        print("  → Usando modelo sintético.")

    # ── Fallback: modelo sintético ────────────────────────────────
    margin = 0.25
    Ws, Es = W-margin, E+margin
    Ss, Ns = S-margin, N+margin
    res = 500
    x = np.linspace(Ws, Es, res); y = np.linspace(Ss, Ns, res)
    X, Y = np.meshgrid(x, y)
    np.random.seed(42)
    c2d = np.array([[coast_lon(la)]*res for la in y])
    d = X - c2d
    z = np.where(d>=0, 0,
        np.where(d>=-0.04, d/0.04*80,
        np.where(d>=-0.22, -80+(d+0.04)/0.18*2680,
                 -2760+(d+0.22)*800)))
    cord = 820*np.exp(-((d-0.07)**2)/(2*0.038**2))
    z += np.where(d>0, cord, 0)
    for lv, dv in [(-29.90,-200), (-29.75,-90)]:
        m = np.exp(-((Y-lv)**2)/(2*0.020**2))
        z += np.where(d>0.02, dv*m*np.clip((d-0.02)/0.08,0,1), 0)
    def n(s,a): return gaussian_filter(np.random.randn(res,res),s)*a
    noise = n(60,170)+n(18,55)+n(6,20)+n(2,8)
    ruf = np.exp(np.clip(d/0.04,-4,0))*(1-np.exp(np.clip(d/0.20,-4,0)))
    z += noise*(ruf*0.8+np.clip(d/0.05,0,1)*0.5)
    return X, Y, z


def draw_isobatas(ax, ext, proj):
    """
    Dibuja isobatas tenues sobre el mapa de zoom.
    Usa batimetría real (ETOPO1) si está disponible, o el modelo sintético.
    Añade etiquetas en las isobatas más importantes (-200 y -1000).
    """
    LON, LAT, Z = get_bathy_grid(ext)

    # Solo trazamos en la zona del mapa (enmascarar tierra)
    Z_sea = np.where(Z < 0, Z, np.nan)

    for iso, lw, ls in zip(ISOBATAS, ISO_LW, ISO_LS):
        cs = ax.contour(LON, LAT, Z_sea,
                        levels=[iso],
                        colors=[ISO_COLOR],
                        linewidths=[lw],
                        linestyles=[ls],
                        alpha=ISO_ALPHA,
                        transform=proj,
                        zorder=6)
        # Etiqueta solo en -200 m y -1000 m
        if iso in [-200, -1000]:
            ax.clabel(cs,
                      fmt={iso: f"{abs(iso)} m"},
                      fontsize=5.5,
                      inline=True,
                      inline_spacing=3,
                      colors=ISO_COLOR,
                      zorder=6)
            # Ajustar alpha de las etiquetas manualmente
            for txt in cs.labelTexts:
                txt.set_alpha(ISO_ALPHA + 0.25)   # un poco más visibles que las líneas
                txt.set_path_effects([stroke(1.0)])


# ── COLORMAPA GEBCO (fallback) ────────────────────────────────────────────────
def gebco_cmap():
    s=[(0.00,"#040428"),(0.08,"#081550"),(0.18,"#0e4490"),
       (0.30,"#1a6db8"),(0.40,"#3a98ce"),(0.46,"#70c0e0"),
       (0.49,"#aadce8"),(0.498,"#c8eef0"),(0.500,"#d8f0d0"),
       (0.506,"#b0d898"),(0.53,"#88c070"),(0.60,"#a8b860"),
       (0.68,"#c8a058"),(0.78,"#d4b070"),(0.88,"#dcc490"),(1.00,"#e8d8b0")]
    return LinearSegmentedColormap.from_list("gebco_hr",s)

def zoom_bg(ax, ext, proj):
    """Fondo batimétrico sintético (fallback sin tiles)."""
    LON, LAT, z = get_bathy_grid(ext)
    norm_z = (np.clip(z,-3000,1000)-(-3000))/(1000-(-3000))
    rgb = gebco_cmap()(norm_z)[:,:,:3]
    ls_hs = LightSource(azdeg=295, altdeg=42)
    c2d = np.array([[coast_lon(la)]*LON.shape[1] for la in LAT[:,0]])
    d = LON - c2d
    hs = ls_hs.hillshade(z, vert_exag=0.015, dx=0.001, dy=0.001)
    sea_mask = np.clip(-d/0.02, 0, 1)
    hs_blend = (0.45+0.55*hs)*sea_mask + 1.0*(1-sea_mask)
    rgb = np.clip(rgb*hs_blend[:,:,np.newaxis], 0, 1)
    W,E,S,N = ext[0]-0.25, ext[1]+0.25, ext[2]-0.25, ext[3]+0.25
    ax.imshow(rgb, extent=[W,E,S,N], origin="lower", transform=proj,
              zorder=1, interpolation="bilinear", aspect="auto")

# ── GRILLA ────────────────────────────────────────────────────────────────────
def add_gl(ax, left=True, right=False, bottom=True, sz=7.5):
    g = ax.gridlines(draw_labels=True, linewidth=0.35, color="white", alpha=0.35,
                     linestyle="--", x_inline=False, y_inline=False)
    g.top_labels=False; g.right_labels=right
    g.left_labels=left; g.bottom_labels=bottom
    g.xlabel_style={"size":sz,"color":FG}
    g.ylabel_style={"size":sz,"color":FG}

# ── FIGURA ────────────────────────────────────────────────────────────────────
def graficar(output="mapa_boyas.png"):
    proj = ccrs.PlateCarree()
    print("Verificando tiles externos…")
    have_tiles = tiles_ok("ocean")
    print(f"  → Tiles disponibles: {have_tiles}")

    fig = plt.figure(figsize=(10,7), facecolor=BG)
    gs = GridSpec(1,2,figure=fig,left=0.03,right=0.97,top=0.92,bottom=0.06,
                  wspace=0.01,width_ratios=[1,2.2])
    ax1 = fig.add_subplot(gs[0], projection=proj)
    ax2 = fig.add_subplot(gs[1], projection=proj)
    for ax in (ax1,ax2):
        ax.set_facecolor("#0c2240")
        for sp in ax.spines.values():
            sp.set_edgecolor(ACCENT); sp.set_linewidth(1.2)

    # ── PANEL IZQUIERDO: Chile ────────────────────────────────────
    ax1.set_extent(CHILE_EXT, crs=proj)
    if have_tiles:
        add_tiles(ax1, "shaded", zoom=4, alpha=0.9)
    else:
        W,E,S,N = CHILE_EXT
        oc = np.linspace(0,1,300).reshape(1,-1); oc = np.tile(oc,(300,1))
        oc_cmap = LinearSegmentedColormap.from_list("oc",
            [(0,"#091828"),(0.5,"#0d2e55"),(1,"#1a4c82")])
        ax1.imshow(oc, extent=[W,E,S,N], origin="lower", transform=proj,
                   cmap=oc_cmap, alpha=0.9, zorder=1, interpolation="bilinear", aspect="auto")
        LAND50 = cfeature.NaturalEarthFeature("physical","land","50m")
        ax1.add_feature(LAND50, facecolor="#d0b470", alpha=0.85, zorder=2)

    C50  = cfeature.NaturalEarthFeature("physical","coastline","50m")
    B110 = cfeature.NaturalEarthFeature("cultural","admin_0_countries","110m")
    ax1.add_feature(C50,  linewidth=0.65, edgecolor="white", facecolor="none", zorder=4)
    ax1.add_feature(B110, linewidth=0.35, edgecolor="#cccccc", facecolor="none",
                    linestyle="--", zorder=4)

    rw=ZOOM_EXT[1]-ZOOM_EXT[0]; rh=ZOOM_EXT[3]-ZOOM_EXT[2]
    ax1.add_patch(mpatches.Rectangle((ZOOM_EXT[0],ZOOM_EXT[2]),rw,rh,
        linewidth=0, facecolor=RECT_C, alpha=0.30, transform=proj, zorder=6))
    ax1.add_patch(mpatches.Rectangle((ZOOM_EXT[0],ZOOM_EXT[2]),rw,rh,
        linewidth=2.2, edgecolor=RECT_C, facecolor="none", transform=proj, zorder=7))

    ax1.plot(COQUIMBO["lon"],COQUIMBO["lat"], marker="o", color=REF_C,
             ms=5, transform=proj, zorder=9)
    ax1.text(COQUIMBO["lon"]+0.35, COQUIMBO["lat"]-0.3, "Coquimbo",
             color=REF_C, fontsize=6.5, transform=proj, zorder=9,
             path_effects=[stroke(1.5)])
    add_gl(ax1, left=True, bottom=True, sz=6.5)

    # ── PANEL DERECHO: Zoom ───────────────────────────────────────
    ax2.set_facecolor(PANEL)
    ax2.set_extent(ZOOM_EXT, crs=proj)
    if have_tiles:
        add_tiles(ax2, "ocean",  zoom=10, alpha=1.0)
        add_tiles(ax2, "shaded", zoom=10, alpha=0.28)
    else:
        zoom_bg(ax2, ZOOM_EXT, proj)

    # Punto de pronóstico
    ax2.scatter(lon_pronostico, lat_pronostico, transform=proj,
                marker="D", s=70, color=PRON_C,
                edgecolors="white", linewidths=0.8, zorder=10)

    # Boyas 1–6
    ax2.scatter(lons_boyas, lats_boyas, transform=proj,
                marker="^", s=75, color=BOYA_C,
                edgecolors="white", linewidths=0.8, zorder=10)
    for num,lo,la in zip(numeros_boyas,lons_boyas,lats_boyas):
        ax2.text(lo+0.005, la+0.005, str(num), color="white", fontsize=5.5,
                 transform=proj, zorder=11, path_effects=[stroke(1.4)])

    # Coquimbo
    ax2.plot(COQUIMBO["lon"],COQUIMBO["lat"], transform=proj,
             marker="*", color=REF_C, ms=12,
             markeredgecolor="white", mew=0.6, zorder=10)
    ax2.text(COQUIMBO["lon"]+0.005, COQUIMBO["lat"]-0.014,
             COQUIMBO["nombre"], color=REF_C, fontsize=7.5, fontweight="bold",
             transform=proj, zorder=11, path_effects=[stroke(1.5)])

    # Leyenda
    hb = mlines.Line2D([],[],color=BOYA_C,marker="^",ls="none",ms=8,
                       mec="white",mew=0.7,label="Boya")
    hp = mlines.Line2D([],[],color=PRON_C,marker="D",ls="none",ms=7,
                       mec="white",mew=0.7,label="Pto. pronóstico")
    hr = mlines.Line2D([],[],color=REF_C,marker="*",ls="none",ms=10,
                       mec="white",mew=0.5,label=COQUIMBO["nombre"])
    ax2.legend(handles=[hb,hp,hr], loc="lower right", fontsize=8.5,
               framealpha=0.80, facecolor="#0d1b2a", edgecolor=ACCENT, labelcolor=FG)

    add_gl(ax2, left=False, right=True, bottom=True, sz=7.5)

    # Escala 10 km exactos a lat -30°
    scale_deg = 10.0 / (np.cos(np.radians(-30.0)) * 111.32)
    sx = ZOOM_EXT[0]+0.025; sy = ZOOM_EXT[2]+0.025
    ax2.plot([sx, sx+scale_deg],[sy,sy], transform=proj,
             color="white", lw=3, solid_capstyle="butt", zorder=12)
    for dx in [0, scale_deg]:
        ax2.plot([sx+dx]*2,[sy-0.003,sy+0.003], transform=proj,
                 color="white", lw=1.5, zorder=12)
    ax2.text(sx+scale_deg/2, sy+0.008, "10 km", color="white", fontsize=6.5,
             ha="center", transform=proj, zorder=12, path_effects=[stroke(1.2)])

    fig.suptitle("Red de Boyas \u2014 Costa de Coquimbo, Chile\n",
                 fontsize=13.5, color=FG, fontweight="bold", y=0.98)

    fig.subplots_adjust(left=0.03, right=0.97, top=0.92, bottom=0.06, wspace=0.01)
    plt.savefig(output, format="png", dpi=500, facecolor=fig.get_facecolor())
    print(f"\u2713 Figura guardada: '{output}'")
    plt.show()

if __name__ == "__main__":
    graficar(output="boyas.png")

Verificando tiles externos…
  → Tiles disponibles: True
✓ Figura guardada: 'boyas.png'
